# Exploratory Data Analysis (EDA)

Telco Customer Churn dataset – initial exploration.


In [ ]:
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import os

import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("../notebooks/figures", exist_ok=True)

In [ ]:
# Load processed data (generated by data pipeline)
processed_path = "../data/processed/telco_churn_processed.parquet"
df = pd.read_parquet(processed_path)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Missing values per column
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Duplicate analysis
print("Full-row duplicates (raw CSV had unique customerID):", df.duplicated().sum())
# Note: After dropping customerID, some feature rows may be identical
# This is acceptable for ML - we do not drop during EDA
feature_dupes = df.duplicated().sum()
print(f"Feature duplicates (after dropping customerID): {feature_dupes}")

In [ ]:
# Target distribution
target_counts = df["Churn"].value_counts()
print(target_counts)
target_pct = df["Churn"].value_counts(normalize=True) * 100
print(target_pct.round(2).astype(str) + "%")
ax = target_counts.plot(kind="bar")
plt.title("Churn distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.tight_layout()
fig_path = "../notebooks/figures/churn_distribution.png"
plt.savefig(fig_path, dpi=150)
print(f"Saved: {fig_path}")
plt.close()

In [ ]:
# Numeric feature summary
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", num_cols)
print(df[num_cols].describe())

In [ ]:
# Categorical feature summary
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Categorical columns:", cat_cols)
for col in cat_cols:
    print(f"\n=== {col} ===")
    print(f"Unique values: {df[col].nunique()}")
    vc = df[col].value_counts()
    print(vc.head(10))
    if len(vc) > 10:
        print(f"... and {len(vc) - 10} more values")

In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numeric features)")
plt.tight_layout()
fig_path = "../notebooks/figures/correlation_heatmap.png"
plt.savefig(fig_path, dpi=150)
print(f"Saved: {fig_path}")
plt.close()

In [ ]:
# Churn rate by key categorical features
key_cats = ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen"]
for col in key_cats:
    if col in df.columns:
        print(f"\n=== Churn rate by {col} ===")
        crosstab = pd.crosstab(df[col], df["Churn"], normalize="index") * 100
        print(crosstab.round(2))